# Text Summarization Pipeline: Pre-Processing, Aggregation, and Prompt Architecture

This notebook provides a complete technical guide to the **Text Summarization Pipeline** ([`T1 - Text Summarization`](.)), covering the end-to-end process through which raw congressional hearing transcripts are cleaned, pre-processed, partitioned via turn-aware chunking, aggregated via hierarchical Map-Reduce LLM passes, and consolidated into production databases.

--- 

## 1. End-to-End Workflow & Output Flowchart

### Pipeline Flowchart

```mermaid
flowchart TD
    subgraph S1 ["Step 1: Document Ingestion & Turn Cleaning"]
        A["GovInfo CHRG Transcripts<br/>(Parquet: turns array)"] --> B["Filter non-spoken Q&A ('qfr')<br/>Format speaker labels"]
        B --> C["Clean Spoken Turn Stream"]
    end

    subgraph S2 ["Step 2: Turn-Aware Token Chunking"]
        C --> D["chunking.py (chunk_turns)"]
        D --> E["Greedily pack whole turns<br/>(~4,000 token budget)"]
        E --> F["Turn-Aware Chunks<br/>(text, n_turns, n_tokens, speakers)"]
    end

    subgraph S3 ["Step 3: Map Stage (LLM Pass)"]
        F --> G["MAP PROMPT (summarize_chunk)"]
        G --> H["Segment Summaries<br/>(~150 words each)"]
    end

    subgraph S4 ["Step 4: Collapse Stage (Conditional / Recursive)"]
        H --> I{"Combined Tokens<br/>> Budget?"}
        I -- "Yes (> 4,000 tokens)" --> J["COLLAPSE PROMPT (collapse_summaries)<br/>Condense run of segment summaries"]
        J --> K["Intermediate Condensed Summaries<br/>(~200 words each)"]
        K --> I
    end

    subgraph S5 ["Step 5: Reduce Stage (Final LLM Pass)"]
        I -- "No (Fits in context)" --> L["REDUCE PROMPT (reduce_summaries)"]
        L --> M["Final Attributed Hearing Summary<br/>(200–250 words, <= 512 tokens)"]
    end

    subgraph S6 ["Step 6: Production Execution & Consolidation"]
        M --> N["10-Shard Parallel Batch (T1.2.1)"]
        N --> O["Consolidation & Verification (T1.2.2)<br/>(Assert zero duplicates)"]
        O --> P["hearing_summaries.parquet<br/>status/*.jsonl CallLog Provenance"]
    end

    subgraph S7 ["Step 7: Quality Assessment & Evaluation"]
        P --> Q["Match with C-SPAN Abstracts (T1.3.1)"]
        Q --> R["BGE Embedding & Cosine Similarity (T1.3.2)"]
        R --> S["Discriminant Validation<br/>Same Doc: 0.742 vs Diff Comm: 0.384"]
    end
```

### Pipeline Stages and Output Matrix

| Stage | Module / Script | Primary Operations | Input Data | Generated Output Artifact & Type |
|---|---|---|---|---|
| **1. Ingestion & Cleaning** | `chunking.py` | Drops written Q&A (`qfr`); standardizes honorifics/surnames (`The Chairman`, etc.). | Raw GovInfo `hearings_sst.parquet` (`turns` dict list) | **Cleaned Turn Stream** (`list[dict]` with `honorific`, `surname`, `text`, `spoken`) |
| **2. Turn-Aware Chunking** | `chunking.py` (`chunk_turns`) | Packs whole turns greedily into ~4k token budget; splits oversized monologues on `\n\n`. | Cleaned Turn Stream | **Turn-Aware Chunks** (`list[dict]`: `text`, `n_turns`, `n_tokens`, `speakers`, `split_turn`) |
| **3. Map Stage** | `summarize.py` (`summarize_chunk`) | Extracts claims, speaker positions, dollar figures, and statutes per chunk. | Turn-Aware Chunks | **Segment Summaries** (`list[str]`, ~150 words per chunk) |
| **4. Collapse Stage** | `summarize.py` (`collapse_summaries`) | Recursively condenses batches of segment summaries without drawing global conclusions. | Batched Segment Summaries (>4,000 tokens) | **Intermediate Condensed Summaries** (`list[str]`, ~200 words each) |
| **5. Reduce Stage** | `summarize.py` (`reduce_summaries`) | Synthesizes hearing into continuous prose; applies `trim_to_sentence()` sentence boundary cap. | All segment / collapsed summaries | **Final Hearing Summary** (`str`, 200–250 words, $\le 512$ tokens) |
| **6. Production Sharding** | `T1.2.1` & `T1.2.2` | Runs 10 disjoint shards (`iloc[s::10]`); consolidates with duplicate & completeness checks. | 2,352 SST Hearings | **`hearing_summaries.parquet`** (Master DB) + **`status/{pkg}.jsonl`** (Call logs) |
| **7. Quality Assessment** | `T1.3.1` & `T1.3.2` | Matches with C-SPAN abstracts; generates 1,024-d BGE embeddings; computes cosine matrix. | `hearing_summaries.parquet` + C-SPAN abstracts | **`cspan_matches.parquet`** + **`cosine_similarity_averages.pdf`** |

--- 

## 2. Document Pre-Processing & Turn-Aware Splitting

### Why Turn-Aware Splitting?
Congressional hearings are multi-party deliberative proceedings. Dividing transcripts arbitrarily by fixed character or token windows splits individual arguments in half across separate LLM calls, causing the model to lose speaker identity, context, and attribution.

### Core Mechanics in `chunking.py`:
1. **Speaking Turn as Atomic Unit**: A speaking turn represents one participant's uninterrupted contribution. Turns are packed whole into chunks greedily until the token budget is reached.
2. **Exclusion of Written Submissions (`qfr`)**: Turns flagged as `section == 'qfr'` (Questions for the Record) are omitted because they represent written post-hearing Q&A rather than spoken testimony.
3. **Speaker Label Formatting (`format_turn`)**:
   - Format: `"{honorific} {surname}: {text}"` (e.g., `"Senator Warner: ..."`).
   - If no surname exists (common for presiding officers), it formats as `"The Chairman"` to prevent the model from guessing or misattributing statements to an unknown speaker.
4. **Exact Token Budgeting**:
   - Uses the model's actual tokenizer (`tokenizer.encode(..., add_special_tokens=False)`) rather than rough character multipliers.
   - Explicitly accounts for separator tokens (`join_cost = len(tokenizer.encode('\n\n'))`).
5. **Over-Long Turn Handling (`_split_long_turn`)**:
   - If a single monologue exceeds the entire chunk budget (<0.3% of turns), it is divided along **paragraph boundaries** (`\n\n`), never mid-sentence.
   - Re-encodes each slice to ensure BPE token merges do not overflow the budget.

In [ ]:
# Demonstrating Turn Formatting & Chunk Packing Logic from T1.X/chunking.py
from typing import Any, Iterable

def format_turn(turn: dict[str, Any]) -> str:
    """Format turn with speaker label and spoken text."""
    name = turn.get("surname") or ""
    honorific = turn.get("honorific") or ""
    who = f"{honorific} {name}".strip() or "Unidentified speaker"
    if turn.get("section") == "qfr":
        who = f"{who} (submitted for the record)"
    return f"{who}: {turn['text']}"

# Sample hearing turns
sample_turns = [
    {"honorific": "The", "surname": "Chairman", "text": "The committee will come to order. Today we review NASA's deep space exploration systems.", "section": "spoken"},
    {"honorific": "Mr.", "surname": "Nelson", "text": "Thank you Mr. Chairman. Commercial lunar payload services and Artemis III milestones remain on schedule.", "section": "spoken"},
    {"honorific": "Dr.", "surname": "Smith", "text": "Written questions regarding propulsion efficiency.", "section": "qfr"}
]

print("--- Formatted Spoken Turns ---")
for t in sample_turns:
    if t.get("section") != "qfr":
        print(format_turn(t))


--- 

## 3. Hierarchical Aggregation Architecture (Map-Reduce with Recursive Collapse)

To summarize long transcripts while preserving fine-grained positions and specific figures, the pipeline applies a **3-stage Map-Reduce process** ([`summarize.py`](file:///Users/milanmiric/Documents/Research/Active%20Papers/LLM%20Text%20Sumarization%20and%20Data%20Extraction/T1%20-%20Text%20Summarization/T1.X%20-%20Background%20Helper%20Functions/summarize.py)):

| Stage | Function | Input Text | Target Output | Purpose |
|---|---|---|---|---|
| **`map`** | `summarize_chunk()` | Raw transcript chunk (~4k tokens) | ~150 words (max 320 tokens) | Read speech; extract subjects, speaker claims, numbers, bills. |
| **`collapse`** | `collapse_summaries()` | Run of intermediate summaries | ~200 words (max 384 tokens) | Condense intermediate summaries when total tokens exceed budget; avoid false conclusions. |
| **`reduce`** | `reduce_summaries()` | All segment summaries combined | 200–250 words (max 512 tokens) | Synthesize single hearing summary with speaker positions and opposing views. |

### The Role of Recursive Collapse
- Short hearings (few chunks) fit directly into the context window and proceed directly from `map` $\to$ `reduce`.
- Long hearings (e.g. 20–50 chunks) produce hundreds of intermediate summary lines. When concatenated intermediate summaries exceed `budget_tokens`, the pipeline automatically batches them and executes 1 or more `collapse` rounds before the final `reduce` pass.

--- 

## 4. LLM Prompts & Exact Templates

Using a single generic prompt across map, collapse, and reduce produces severe failure modes (such as attributing positions to whoever was mentioned last in a summary). The pipeline therefore uses **three distinct, specialized prompts**:

### 4.1. Map Stage Prompt (Chunk-Level Extraction)

**System Prompt:**
> `You are an expert in text summarization with a working knowledge of United States congressional hearings. You report only what the transcript states.`

**User Prompt Template (`MAP_USER`):**
```text
Below is one consecutive segment of a congressional hearing transcript. Each speaking turn is labelled with the speaker.

Summarize this segment in about 150 words. Cover:
- the subjects raised, and what was said about them;
- the position each named speaker takes, attributed to that speaker;
- any numbers, dollar amounts, dates, and bill or act names, copied exactly as they appear.

If the segment is procedural or ceremonial and contains none of these, say so in one sentence.

--- SEGMENT ---
{text}
--- END OF SEGMENT ---

Write only the summary of the segment above.
```

--- 

### 4.2. Collapse Stage Prompt (Intermediate Condensation)

**System Prompt:**
> `You are an expert in text summarization with a working knowledge of United States congressional hearings. You condense summaries without adding to them.`

**User Prompt Template (`COLLAPSE_USER`):**
```text
Below are summaries of consecutive segments of one congressional hearing, in the order they occurred. They are already summaries, not transcript.

Condense them into one account of about 200 words that keeps the named speakers, their positions, and the numbers and legislation. This is a middle portion of the hearing, so do not write an overall conclusion about the hearing.

--- SEGMENT SUMMARIES ---
{text}
--- END ---

Write only the condensed account.
```

--- 

### 4.3. Reduce Stage Prompt (Whole-Hearing Synthesis)

**System Prompt:**
> `You are an expert in text summarization with a working knowledge of United States congressional hearings. You write from the summaries given and add nothing that is not in them.`

**User Prompt Template (`REDUCE_USER`):**
```text
Below are summaries of every segment of one congressional hearing, in order. They are already summaries, not transcript.

Write a single summary of the hearing as a whole, in about {target_words} words. Cover:
- what the hearing was about;
- the main subjects, the most heavily discussed first;
- the position each named speaker took;
- any point where speakers took opposing positions, naming both sides;
- the key numbers and the legislation referred to, exactly as given.

Where the segment summaries do not establish one of these, leave it out rather than inferring it.

--- SEGMENT SUMMARIES ---
{text}
--- END ---

Write only the summary of the hearing, in about {target_words} words, as continuous prose.
```
*Note: `{target_words}` is passed dynamically as `"200 to 250"`.*

--- 

## 5. Technical Safeguards & Generation Hyperparameters

### 1. Deterministic Sampling Configuration
- `do_sample: True`, `temperature: 0.1`
- `top_p` is deliberately omitted to prevent non-standard hyperparameters from undermining corpus reproducibility.
- Near-greedy temperature ensures strict factual grounding on input text.

### 2. Disabling Thinking Mode for Reasoning Models
- For models such as `Qwen/Qwen3.5-4B`, `enable_thinking=False` is passed to `tokenizer.apply_chat_template()`.
- This prevents `<think>` tokens from consuming the allocated completion budget.

### 3. Clean Sentence Truncation (`trim_to_sentence`)
- If generation hits the token cap (e.g. 512 tokens), the text is cleanly trimmed back to the last complete sentence terminating punctuation (`.!?`, handling trailing quotes/brackets).
- Prevents broken, dangling half-clauses from being embedded by downstream topic models.

### 4. Full Provenance Logging (`CallLog`)
- Every single LLM invocation writes a JSONL record to disk containing:
  - `document_id`, `model`, `prompt_version`
  - `stage` (`map`, `collapse`, `reduce`), `level`, `index`
  - `input_sha256` (SHA-256 digest of input prompt)
  - `input_tokens`, `output_tokens`, `seconds`
  - **Full unedited generated output text**.

In [ ]:
# Demonstrating Sentence Trimming and CallLog Metadata Structure
import hashlib
import json

_SENTENCE_END = tuple('.!?') + tuple(f'{p}{q}' for p in '.!?' for q in '"\')]')

def trim_to_sentence(text: str) -> str:
    """Cut back to the last complete sentence when token cap is reached."""
    text = text.rstrip()
    if not text or text.endswith(_SENTENCE_END):
        return text
    cut = max(text.rfind(mark) for mark in '.!?')
    if cut <= 0:
        return text
    end = cut + 1
    while end < len(text) and text[end] in '"\')]':
        end += 1
    return text[:end].rstrip()

truncated_sample = "Senator Warner highlighted NASA funding constraints and requested an audit of commercial vendors. The committee agreed to"
print("Raw Truncated Text:", truncated_sample)
print("Trimmed Clean Text:", trim_to_sentence(truncated_sample))


--- 

## 6. Execution Stages and Quality Assessment

### Step 1: Model Selection & Prototyping ([`T1.1`](file:///Users/milanmiric/Documents/Research/Active%20Papers/LLM%20Text%20Sumarization%20and%20Data%20Extraction/T1%20-%20Text%20Summarization/T1.1%20-%20Process%20Prototyping%20and%20Evaluation))
- Evaluated 4 candidate open-weights models (`Qwen/Qwen3.5-4B`, `Gemma-4-E4B-it`, `Llama-3.1-8B`, `Llama-3.2-3B`).
- **`Qwen/Qwen3.5-4B`** (in unquantized 16-bit `bf16`) was selected for production: produced **57 checkable claims** per sample vs 11–38 for alternatives, while maintaining high speaker attribution accuracy.

### Step 2: Production Sharded Batch Execution ([`T1.2`](file:///Users/milanmiric/Documents/Research/Active%20Papers/LLM%20Text%20Sumarization%20and%20Data%20Extraction/T1%20-%20Text%20Summarization/T1.2%20-%20Process%20Execution))
- Executed over **2,352 hearings** (93.6 million total words) using 10 disjoint parallel shards (`iloc[s::10]`).
- Shards consolidated into master table `hearing_summaries.parquet` with assertions ensuring zero duplicates and 100% completeness.

### Step 3: Independent Quality Evaluation ([`T1.3`](file:///Users/milanmiric/Documents/Research/Active%20Papers/LLM%20Text%20Sumarization%20and%20Data%20Extraction/T1%20-%20Text%20Summarization/T1.3%20-%20Quality%20Assessment))
- Evaluated against independent human-written C-SPAN editorial program abstracts via 1,024-d dense vector embeddings (`bge-large-en-v1.5`).
- **3-Way Cosine Discriminant Analysis**:
  1. **Same Document** (LLM Summary $\leftrightarrow$ C-SPAN Abstract): Mean Cosine Similarity = **0.742**
  2. **Same Committee, Different Hearing**: Mean Cosine Similarity = **0.518**
  3. **Different Committee**: Mean Cosine Similarity = **0.384**
- Confirms statistically that generated summaries capture granular, hearing-specific content rather than generic committee boilerplate.